In [1]:
import torch
import torch.distributions as dist
import math

import matplotlib.pyplot as plt
from tqdm import tqdm
import utilities  

class SimpleCMAES:
    """
    一个基于 PyTorch 原生实现的简化版 CMA-ES (Covariance Matrix Adaptation Evolution Strategy) 求解器。
    
    CMA-ES 是一种用于连续优化问题的进化算法。它的核心思想是：
    1. 维护一个多维正态分布（由均值 Mean 和 协方差矩阵 Covariance 决定）。
    2. 从这个分布中采样出一组解（种群）。
    3. 根据解的好坏（适应度），调整分布的均值（向好解移动）和协方差矩阵（改变搜索形状），
       使得下一次采样更有可能产生更好的解。
    """
    
    def __init__(self, num_params, population_size, init_mean, init_sigma, device='cuda'):
        """
        初始化 CMA-ES 求解器。

        Args:
            num_params (int): 优化变量的维度 D (例如 2 * N_chiplets)。
            population_size (int): 种群大小 lambda (例如 50)。
            init_mean (torch.Tensor): 初始均值向量，形状 [D]。
            init_sigma (float): 初始步长 (标准差)，控制初始搜索范围的大小。
            device (str): 运行设备 ('cpu' 或 'cuda')。
        """
        self.num_params = num_params
        self.pop_size = population_size
        self.sigma = init_sigma
        self.device = device

        # 1. 初始化均值 (Mean)
        # 均值代表了当前搜索分布的中心点，也就是我们认为目前"最好"的位置
        if isinstance(init_mean, torch.Tensor):
            self.mean = init_mean.clone().to(device).float()
        else:
            self.mean = torch.tensor(init_mean, device=device, dtype=torch.float32)

        # 2. 初始化协方差矩阵 (Covariance Matrix)
        # 协方差矩阵决定了搜索分布的"形状"（是圆的还是扁的，朝哪个方向倾斜）。
        # 初始时我们不知道哪个方向好，所以设为单位矩阵 (Identity Matrix)，代表各向同性的搜索（正圆形/正球体）。
        self.cov = torch.eye(num_params, device=device, dtype=torch.float32)

        # 3. 初始化精英选择参数 (Selection Parameters)
        # 我们只选择前 mu 个最好的个体来更新分布
        self.mu = population_size // 2  
        
        # 4. 初始化权重 (Weights)
        # 我们不只是简单平均最好的 mu 个解，而是给排名靠前的解更大的权重。
        # 公式：w_i = ln(mu + 0.5) - ln(i)
        # 这种对数递减的权重分配让最好的解（Rank 1）对均值移动的影响最大。
        weights_prime = torch.log(torch.tensor(self.mu + 0.5, device=device)) - \
                        torch.log(torch.arange(1, self.mu + 1, device=device, dtype=torch.float32))
        
        # 归一化权重，使得所有权重之和为 1
        self.weights = weights_prime / torch.sum(weights_prime)
        
        # 5. 计算学习率 (Learning Rate for Covariance Update)
        # c_mu 是控制协方差矩阵更新速度的系数。
        # 为了简化实现，这里使用一个基于 mu_eff 的经验公式。
        mu_eff = 1.0 / torch.sum(self.weights ** 2)
        self.c_mu = min(1.0, mu_eff / (num_params ** 2))
        
        # 6. 数值稳定性参数
        self.epsilon = 1e-8  # 防止矩阵奇异（不可逆）的微小值

    def ask(self):
        """
        【生成】阶段：根据当前分布生成下一代候选解。
        
        Returns:
            solutions (torch.Tensor): 形状 [pop_size, num_params]，即生成的种群。
        """
        # 构建多维正态分布 Multivariate Normal Distribution
        # 均值 = self.mean
        # 协方差 = (sigma^2) * self.cov
        # 注意：为了数值稳定，我们在 cov 对角线上加上微小的 epsilon
        
        # 计算实际用于采样的协方差矩阵：Sigma^2 * C
        sample_cov = (self.sigma ** 2) * self.cov + self.epsilon * torch.eye(self.num_params, device=self.device)
        
        # 创建分布对象
        mvn = dist.MultivariateNormal(self.mean, covariance_matrix=sample_cov)
        
        # 采样种群
        # sample() 会返回 [pop_size, num_params] 的张量
        solutions = mvn.sample((self.pop_size,))
        
        return solutions

    def tell(self, solutions, scores):
        """
        【更新】阶段：根据种群的评估得分，更新分布参数（均值、协方差、步长）。
        
        Args:
            solutions (torch.Tensor): ask() 生成的种群，形状 [pop_size, num_params]。
            scores (torch.Tensor): 对应的能量值/损失值，形状 [pop_size]。越小越好。
        """
        # 确保数据在正确的设备上
        solutions = solutions.to(self.device)
        scores = scores.to(self.device)

        # 1. 排序 (Sorting)
        # 根据能量值从小到大排序 (因为我们的目标是最小化)
        # argsort 返回的是排序后的索引
        sorted_indices = torch.argsort(scores)
        
        # 2. 选择精英 (Selection)
        # 选出表现最好的前 mu 个解
        top_indices = sorted_indices[:self.mu]
        elites = solutions[top_indices]  # 形状: [mu, num_params]
        
        # 保存旧的均值，后面更新协方差时需要用到
        old_mean = self.mean.clone()

        # 3. 更新均值 (Update Mean)
        # 新的均值 = 精英解的加权平均
        # 这一步让搜索中心向表现好的区域移动
        # torch.mv 是矩阵向量乘法: [num_params, mu] @ [mu] -> [num_params]
        # 注意：这里需要转置 elites [mu, D] -> [D, mu] 才能和 weights [mu] 相乘
        self.mean = torch.mv(elites.T, self.weights)

        # 4. 更新协方差矩阵 (Update Covariance Matrix - Rank-mu Update)
        # 这是 CMA-ES 的核心。我们希望协方差矩阵能"记住"这次精英解分布的方向。
        # 如果精英解都分布在某个方向上（例如对角线），协方差矩阵就会变形，
        # 使得下一次采样生成的椭圆更长，沿着该方向探索更远。
        
        # 计算“进化路径”向量 y_i = (x_i - m_old) / sigma
        # 这些向量代表了精英解相对于上一代中心的偏移方向
        y_vectors = (elites - old_mean) / self.sigma  # 形状: [mu, num_params]

        # 计算加权协方差
        # 对于每个精英向量 y，计算外积 y * y^T，得到秩-1矩阵，然后加权求和
        # 这一步可以用矩阵乘法高效实现: (weights * y).T @ y
        # weights.view(-1, 1) 将权重变成列向量以便广播
        weighted_cov = torch.matmul((self.weights.view(-1, 1) * y_vectors).T, y_vectors)
        
        # 应用 Rank-mu 更新公式：
        # C_new = (1 - c_mu) * C_old + c_mu * Weighted_Cov
        # (1 - c_mu) * C_old : 保留一部分历史形状（惯性）
        # c_mu * Weighted_Cov : 融入当前代精英的分布形状
        self.cov = (1 - self.c_mu) * self.cov + self.c_mu * weighted_cov

        # 5. 强制协方差矩阵对称 (Symmetry Enforcement)
        # 由于浮点数误差，矩阵可能变得不对称，这会导致数学错误。
        # 强制使其对称：C = (C + C^T) / 2
        self.cov = (self.cov + self.cov.T) / 2

        # 6. 更新步长 (Update Sigma - Simplified)
        # 标准 CMA-ES 使用复杂的路径积分控制步长。
        # 为了简化实现且保证收敛，这里使用简单的指数衰减策略（类似模拟退火的降温）。
        # 每次迭代将步长缩小一点点，使得搜索范围逐渐收窄，精细化搜索。
        self.sigma *= 0.98 
        
        # 可以在这里打印调试信息（可选）
        # print(f"  >> Updated Sigma: {self.sigma:.4f}")

In [2]:

# 假设 calculate_energy 已经定义好并可以被调用
# 在实际使用中，你需要确保这个函数在当前命名空间可见
# from sa_module import calculate_energy (示例)

class FitnessAdapter:
    """
    适应度适配器 (Fitness Adapter)
    
    功能职责：
    1. 充当 CMA-ES 求解器(数学世界)与物理评估函数(物理世界)之间的桥梁。
    2. 将扁平的优化向量 [x1, y1, x2, y2...] 解码为合法的布局张量 [N, 5]。
    3. 强制实施边界约束 (Clamping)，防止芯片跑出画布。
    4. 批量计算整个种群的能量得分。
    """
    
    def __init__(self, chiplets, board_size, device='cuda'):
        """
        初始化适配器，预计算约束条件以加速后续迭代。

        Args:
            chiplets (dict): 芯片字典，格式 {index: {'len': ..., 'wid': ...}, ...}
            board_size (tuple): 画布尺寸 (board_len, board_wid)
            device (str): 计算设备
        """
        self.chiplets = chiplets
        self.device = device
        self.board_len, self.board_wid = board_size
        self.num_chips = len(chiplets)

        # =========================================================
        # 1. 预计算边界约束 (Pre-compute Constraints)
        # =========================================================
        # CMA-ES 生成的坐标可能是任意实数，但芯片必须在画布内。
        # 对于第 i 个芯片，其 x 坐标必须满足: 0 <= x <= Board_L - Chip_L
        # 我们提前算出每个芯片允许的最大 x 和 最大 y，存入 Tensor。
        
        max_coords_list = []  # 存储 [max_x, max_y]
        static_info_list = [] # 存储 [len, wid, index] (这些在优化过程中是不变的)

        # 按索引顺序遍历，确保与 CMA-ES 的向量顺序一致
        for i in range(self.num_chips):
            chip = chiplets[i]
            l, w = chip['len'], chip['wid']
            
            # 计算允许的最大坐标
            max_x = self.board_len - l
            max_y = self.board_wid - w
            max_coords_list.append([max_x, max_y])
            
            # 存储静态信息，后续用于拼接布局
            # 注意：index 转为 float 存储在 Tensor 中
            static_info_list.append([l, w, float(i)])

        # 转为 Tensor 并移动到 GPU，方便后续快速广播计算
        # shape: [N, 2]
        self.max_coords = torch.tensor(max_coords_list, device=device, dtype=torch.float32)
        # shape: [N, 3]
        self.static_info = torch.tensor(static_info_list, device=device, dtype=torch.float32)

    def decode_layout(self, flat_vector):
        """
        核心方法：将 CMA-ES 的扁平向量转化为合法布局。

        Args:
            flat_vector (torch.Tensor): 形状 [2*N]，内容为 [x0, y0, x1, y1, ...]

        Returns:
            layout (torch.Tensor): 形状 [N, 5]，内容为 [[x, y, len, wid, index], ...]
        """
        # 1. Reshape (重塑)
        # 将一维向量 [2*N] 变成 [N, 2]，每行代表一个芯片的 (x, y)
        coords = flat_vector.view(self.num_chips, 2)
        
        # 2. Clamp (截断/限制) - 关键步骤！
        # CMA-ES 可能会生成负数或超大值。
        # 我们利用预计算的 self.max_coords 进行逐元素截断。
        # 逻辑：x = max(0, min(x, max_x))
        # 这一步保证了生成的布局在物理上永远是合法的（不会超出边界）。
        coords = torch.minimum(torch.maximum(coords, torch.zeros_like(coords)), self.max_coords)
        
        # 3. Concatenate (拼接)
        # 将动态优化的坐标 coords [N, 2] 与静态信息 static_info [N, 3] 拼接
        # 结果变成 [N, 5]: (x, y, len, wid, index)
        layout = torch.cat([coords, self.static_info], dim=1)
        
        return layout

    def get_scores(self, solutions, connectivity, weights=None):
        """
        批量计算种群的能量得分。

        Args:
            solutions (torch.Tensor): 种群矩阵，形状 [pop_size, 2*N]
            connectivity (list): 连通性定义
            weights (dict): 能量函数的权重配置

        Returns:
            scores (torch.Tensor): 形状 [pop_size]，每个个体的能量值
        """
        scores_list = []
        
        # 遍历种群中的每一个解
        for i in range(solutions.shape[0]):
            flat_vec = solutions[i]
            
            # 1. 解码：向量 -> 布局
            layout = self.decode_layout(flat_vec)
            
            # 2. 评估：布局 -> 能量
            # 调用外部的 calculate_energy 函数
            # 注意：这个计算比较耗时，是整个算法的瓶颈
            energy = calculate_energy(layout, connectivity, self.chiplets, weights)
            
            scores_list.append(energy)
            
        # 将结果转为 Tensor 返回，方便 CMA-ES 求解器进行排序和更新
        return torch.tensor(scores_list, device=self.device, dtype=torch.float32)

In [3]:
import interconnect_calculator as inter_cal
import temperature_calculator as temp_cal
import chip_temp_pred_models as ctpm

# 自动检测设备
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on device: {DEVICE}")


def get_chiplet_dict_cuda(chip_length, chip_width, Convection_Film_Coefficient, Internal_Heat_Generation_Magnitude):
    """根据 stage2_1_cuda03 口径生成芯粒字典和 A/k 参数。"""
    chip = {
        'len': chip_length,
        'wid': chip_width,
        'CFC': Convection_Film_Coefficient,
        'IHGM': Internal_Heat_Generation_Magnitude,
    }
    chip['A'], chip['k'] = ctpm.get_out_chip_decay_curve_coef(
        chip_len=chip['len'],
        chip_wid=chip['wid'],
        Convection_Film_Coefficient=chip['CFC'],
        Internal_Heat_Generation_Magnitude=chip['IHGM'],
    )
    return chip


def is_2_chip_overlapping_cuda(chip1, chip2, min_edge_dist: float = 0.2):
    """判断两个芯粒是否重叠。"""
    x1, y1, w1, h1 = chip1[0], chip1[1], chip1[2], chip1[3]
    x2, y2, w2, h2 = chip2[0], chip2[1], chip2[2], chip2[3]

    no_overlap = (
        (x1 + w1 + min_edge_dist <= x2)
        or (x2 + w2 + min_edge_dist <= x1)
        or (y1 + h1 + min_edge_dist <= y2)
        or (y2 + h2 + min_edge_dist <= y1)
    )
    return not no_overlap


def check_overlap_within_layout_cuda(layout):
    """检测布局内部是否存在任意重叠。"""
    if not torch.is_tensor(layout):
        layout = torch.tensor(layout, device=DEVICE)
    else:
        layout = layout.to(DEVICE)

    n = len(layout)
    if n <= 1:
        return False

    i_indices, j_indices = torch.triu_indices(n, n, offset=1)
    overlaps = [
        bool(is_2_chip_overlapping_cuda(layout[i], layout[j]))
        for i, j in zip(i_indices, j_indices)
    ]
    return any(overlaps)


def calculate_energy(layout, connectivity, chiplets, weights=None):
    """与 stage2_1_cuda03.ipynb 保持一致的 fitness 口径（越小越好）。"""
    if weights is None:
        weights = {
            'interconnect': 1.0,
            'temp_uniformity': 1.0,
            'max_temp': 0.1,
            'overlap': 10000.0,
            'spread': 0.2,
        }

    if not torch.is_tensor(layout):
        layout = torch.tensor(layout, device=DEVICE, dtype=torch.float32)
    else:
        layout = layout.to(DEVICE)

    inter_connect_length = inter_cal.get_total_interconnect_length_cuda(
        layout=layout,
        connectivity_pairs=connectivity,
    )

    max_temp, temp_uniformity = temp_cal.get_max_temp_and_temp_uniformity_cuda(layout, chiplets)

    overlap_penalty = weights['overlap'] if check_overlap_within_layout_cuda(layout) else 0.0

    coords = torch.stack(
        [
            layout[:, 0] + layout[:, 2] * 0.5,
            layout[:, 1] + layout[:, 3] * 0.5,
        ],
        dim=1,
    )
    centroid = coords.mean(dim=0)
    diff = coords - centroid
    dist_sq = diff[:, 0] ** 2 + diff[:, 1] ** 2
    spread_penalty = dist_sq.sum()

    total_energy = (
        weights['interconnect'] * inter_connect_length
        + weights['temp_uniformity'] * temp_uniformity
        + weights['max_temp'] * max_temp
        + overlap_penalty
        + weights['spread'] * spread_penalty
    )

    return total_energy.item() if torch.is_tensor(total_energy) else float(total_energy)


# ==========================================
# 1. 数据准备（与 stage2_1_cuda03 统一口径）
# ==========================================

def get_fitness_breakdown_cuda(layout, connectivity, chiplets, weights=None):
    """
    ??????? fitness ????????????????????????
    """
    if weights is None:
        weights = {
            'interconnect': 1.0,
            'temp_uniformity': 1.0,
            'max_temp': 0.1,
            'overlap': 10000.0,
            'spread': 0.2,
        }

    if not torch.is_tensor(layout):
        layout = torch.tensor(layout, device=DEVICE, dtype=torch.float32)
    else:
        layout = layout.to(DEVICE).float()

    inter_connect_length = inter_cal.get_total_interconnect_length_cuda(
        layout=layout,
        connectivity_pairs=connectivity,
    )
    max_temp, temp_uniformity = temp_cal.get_max_temp_and_temp_uniformity_cuda(layout, chiplets)

    overlap_raw = 1.0 if check_overlap_within_layout_cuda(layout) else 0.0
    overlap_penalty = weights['overlap'] if overlap_raw > 0 else 0.0

    coords = torch.stack(
        [
            layout[:, 0] + layout[:, 2] * 0.5,
            layout[:, 1] + layout[:, 3] * 0.5,
        ],
        dim=1,
    )
    centroid = coords.mean(dim=0)
    diff = coords - centroid
    dist_sq = diff[:, 0] ** 2 + diff[:, 1] ** 2
    spread_raw = dist_sq.sum()

    def _to_float(x):
        return x.item() if torch.is_tensor(x) else float(x)

    breakdown = {
        'interconnect': {
            'raw': _to_float(inter_connect_length),
            'weight': float(weights['interconnect']),
            'weighted': float(weights['interconnect']) * _to_float(inter_connect_length),
        },
        'temp_uniformity': {
            'raw': _to_float(temp_uniformity),
            'weight': float(weights['temp_uniformity']),
            'weighted': float(weights['temp_uniformity']) * _to_float(temp_uniformity),
        },
        'max_temp': {
            'raw': _to_float(max_temp),
            'weight': float(weights['max_temp']),
            'weighted': float(weights['max_temp']) * _to_float(max_temp),
        },
        'spread': {
            'raw': _to_float(spread_raw),
            'weight': float(weights['spread']),
            'weighted': float(weights['spread']) * _to_float(spread_raw),
        },
        'overlap': {
            'raw': overlap_raw,
            'weight': float(weights['overlap']),
            'weighted': float(overlap_penalty),
        },
    }
    breakdown['total'] = sum(item['weighted'] for item in breakdown.values())
    return breakdown

def build_layout(best_positions, input_data):
    if torch.is_tensor(best_positions):
        rows = best_positions.detach().cpu().tolist()
    else:
        rows = best_positions

    layout = []
    for row_i, row in enumerate(rows):
        x = round(float(row[0]), 4)
        y = round(float(row[1]), 4)

        chip_idx = int(row_i)
        if len(row) >= 5:
            idx_raw = float(row[4])
            idx_int = int(idx_raw)
            if abs(idx_raw - idx_int) <= 1e-6 and 0 <= idx_int < len(input_data):
                chip_idx = idx_int

        if chip_idx < 0 or chip_idx >= len(input_data):
            raise ValueError(f"chip_idx out of range: {chip_idx}, valid [0, {len(input_data)-1}]")

        if len(row) >= 4:
            width = float(row[2])
            height = float(row[3])
        else:
            width = float(input_data[chip_idx][0])
            height = float(input_data[chip_idx][1])

        h = float(input_data[chip_idx][2])
        power_density = float(input_data[chip_idx][3])

        layout.append([
            x,
            y,
            width,
            height,
            h,
            power_density,
            int(chip_idx),
        ])

    for out_row in layout:
        idx = int(out_row[6])
        in_w = float(input_data[idx][0])
        in_h = float(input_data[idx][1])
        out_w = float(out_row[2])
        out_h = float(out_row[3])

        same_order = abs(out_w - in_w) <= 1e-6 and abs(out_h - in_h) <= 1e-6
        swapped_order = abs(out_w - in_h) <= 1e-6 and abs(out_h - in_w) <= 1e-6
        if not (same_order or swapped_order):
            raise ValueError(f"width/height mismatch at chip_idx={idx}: output=({out_w}, {out_h}), input=({in_w}, {in_h})")

    return layout

raw_input = [
    [9, 9, 15, 140000000],
    [6, 6, 15, 300000000],
    [5, 5, 15, 80000000],
    #[4, 4, 15, 120000000],
]

chiplets = {}
for i, d in enumerate(raw_input):
    chiplets[i] = get_chiplet_dict_cuda(
        chip_length=d[0],
        chip_width=d[1],
        Convection_Film_Coefficient=d[2],
        Internal_Heat_Generation_Magnitude=d[3],
    )

connectivity = [
    [(0, 1), 1],
    [(1, 2), 2],
    [(0, 2), 1],
    #[(0, 3), 1],
]

# ==========================================
# 2. 初始化
# ==========================================
avg_len = utilities.avg([c['len'] for c in chiplets.values()])
avg_wid = utilities.avg([c['wid'] for c in chiplets.values()])
num_chips = len(chiplets)
board_len = (avg_len + 10) * num_chips + 10
board_wid = (avg_wid + 10) * num_chips + 10
board_size = (board_len, board_wid)
print(f"Board Size: {board_size}")

adapter = FitnessAdapter(chiplets, board_size, device=DEVICE)

num_params = 2 * num_chips
init_mean = torch.tensor(
    [board_len / 2, board_wid / 2] * num_chips,
    device=DEVICE,
    dtype=torch.float32,
)
init_sigma = 5.0

solver = SimpleCMAES(
    num_params=num_params,
    population_size=50,  # 保留初始候选数量参数
    init_mean=init_mean,
    init_sigma=init_sigma,
    device=DEVICE,
)

# ==========================================
# 3. 优化循环
# ==========================================
max_iterations = 200
energy_history = []
best_energy_overall = float('inf')
best_layout_overall = None

# 在运行 CMA-ES 前统一设置 fitness 权重（后续调参只需改这里）
fitness_weights = {
    'interconnect': 1.0,    # 连线长度权重：越短越好
    'temp_uniformity': 1.0, # 温度均匀性权重：方差越小越好
    'max_temp': 0.1,        # 最大温度权重：最高温越低越好
    'spread': 0.2,          # 分散度权重：鼓励布局更均匀展开
    'overlap': 10000.0      # 重叠惩罚权重：一旦重叠就强力惩罚
}


print("Starting CMA-ES Optimization...")
pbar = tqdm(range(max_iterations), desc="CMA-ES")

for _ in pbar:
    solutions = solver.ask()
    scores = adapter.get_scores(solutions, connectivity, weights=fitness_weights)
    solver.tell(solutions, scores)

    current_best_energy = torch.min(scores).item()
    energy_history.append(current_best_energy)

    if current_best_energy < best_energy_overall:
        best_energy_overall = current_best_energy
        best_idx = torch.argmin(scores)
        best_vec = solutions[best_idx]
        best_layout_overall = adapter.decode_layout(best_vec).detach().clone()

    pbar.set_postfix(
        {
            "Best E": f"{best_energy_overall:.2f}",
            "Curr E": f"{current_best_energy:.2f}",
            "Sigma": f"{solver.sigma:.4f}",
        }
    )

print("Optimization Finished!")


if best_layout_overall is None:
    best_layout_for_output = adapter.decode_layout(solver.mean)
else:
    best_layout_for_output = best_layout_overall

layout = build_layout(best_layout_for_output, raw_input)
print("\nCMA-ES best fitness layout (unified format):")
print("layout = [")
for row in layout:
    print(f"    {row},")
print("]")

# ==========================================
# 4. 结果可视化
# ==========================================
plt.figure(figsize=(10, 6))

history_np = [float(h) for h in energy_history]
best_history = [min(history_np[: i + 1]) for i in range(len(history_np))]

import csv
cmaes_csv_path = "cmaes_fitness_history.csv"
with open(cmaes_csv_path, mode="w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["iteration", "fitness"])
    for iter_idx, score in enumerate(best_history):
        writer.writerow([iter_idx, float(score)])
print(f"CMA-ES CSV exported: {cmaes_csv_path}")

plt.plot(best_history, label='Best Energy', color='green', linewidth=2)
plt.title('CMA-ES Optimization Convergence Curve', fontsize=15)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Energy (Cost)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

final_score = best_history[-1]
plt.scatter(len(best_history) - 1, final_score, color='red', s=50, zorder=5)
plt.text(
    len(best_history) - 1,
    final_score,
    f' Final: {final_score:.4f}',
    fontsize=12,
    color='red',
    verticalalignment='bottom',
)
plt.legend()
plt.show()

final_mean_vec = solver.mean
final_layout = adapter.decode_layout(final_mean_vec)

print("Visualizing Final Layout (from Distribution Mean)...")
try:
    plot_layout = final_layout.detach().cpu() if torch.is_tensor(final_layout) else final_layout
    utilities.show_chip_design_cuda(plot_layout)
except Exception as e:
    print(f"Visualization warning: {e}")

# ==========================================
# 5. 指标报告
# ==========================================
def _to_float(x):
    return x.item() if torch.is_tensor(x) else float(x)

final_breakdown = get_fitness_breakdown_cuda(
    final_layout,
    connectivity,
    chiplets,
    weights=fitness_weights,
)

print("")
print("==================== CMA-ES Final Fitness Breakdown ====================")
metric_items = [
    ('interconnect', 'Interconnect'),
    ('temp_uniformity', 'TempUniformity'),
    ('max_temp', 'MaxTemp'),
    ('spread', 'Spread'),
    ('overlap', 'Overlap'),
]
total_score = final_breakdown['total']
print(f"{'Metric':<15} {'Raw':>12} {'Weight':>12} {'Weighted':>14} {'Ratio(%)':>12}")
for metric_key, metric_name in metric_items:
    item = final_breakdown[metric_key]
    ratio = (item['weighted'] / total_score * 100.0) if abs(total_score) > 1e-12 else 0.0
    print(
        f"{metric_name:<15} {item['raw']:>12.4f} {item['weight']:>12.4f} {item['weighted']:>14.4f} {ratio:>11.2f}%"
    )
print("-" * 78)
total_ratio = 100.0 if abs(total_score) > 1e-12 else 0.0
print(f"{'Total Fitness':<15} {'':>12} {'':>12} {total_score:>14.4f} {total_ratio:>11.2f}%")


g:\VScode\CTAD2\CTAD2_CMA-ES_test\stage2_1\chip_temp_pred_models.py:122: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  inchip_pred_model.load_state_dict(torch.load(inchip_mo

Running on device: cpu
Board Size: (74, 74)
Starting CMA-ES Optimization...


CMA-ES:   0%|          | 0/200 [00:01<?, ?it/s]


TypeError: clamp() received an invalid combination of arguments - got (Tensor, min=float, max=Tensor), but expected one of:
 * (Tensor input, Tensor min = None, Tensor max = None, *, Tensor out = None)
 * (Tensor input, Number min = None, Number max = None, *, Tensor out = None)
